#Paper utilizado:

[1] M. Bahaghighat, M. Ghasemi y F. Ozen,
“A high-accuracy phishing website detection method based on machine learning,”
Journal of Information Security and Applications,
vol. 77,
art. 103553,
2023.
doi: 10.1016/j.jisa.2023.103553.
Available: https://www.sciencedirect.com/science/article/pii/S2214212623001370

In [ ]:
import joblib

log_model = joblib.load("log_model.pkl")
rf_model = joblib.load("rf_model.pkl")
xgb_model = joblib.load("xgb_model.pkl")

label_encoder = joblib.load(
    "label_encoder.pkl"
)

columnas = joblib.load(
    "columnas.pkl"
)

In [ ]:
import pandas as pd
import joblib

log_model = joblib.load("log_model.pkl")
rf_model = joblib.load("rf_model.pkl")
xgb_model = joblib.load("xgb_model.pkl")

label_encoder = joblib.load("label_encoder.pkl")
columnas = joblib.load("columnas.pkl")

print("Los valores -1 son Legit")
print("Los valores 0 son Suspicious")
print("Los valores 1 son Phishing")

campos = {
    "SFH": [-1, 0, 1],
    "popUpWindow": [-1, 0, 1],
    "SSLfinal_State": [-1, 0, 1],
    "Request_URL": [-1, 0, 1],
    "URL_of_Anchor": [-1, 0, 1],
    "web_traffic": [-1, 0, 1],
    "URL_Length": [-1, 0, 1],
    "age_of_domain": [-1, 1],
    "having_IP_Address": [0, 1]
}

entrada = {}

for campo, permitidos in campos.items():
    while True:
        try:
            valor = int(input(f"{campo} {permitidos}: "))
            if valor in permitidos:
                entrada[campo] = valor
                break
            print("Valor inválido")
        except ValueError:
            print("Ingresa un número válido")

nuevo = pd.DataFrame([entrada])

nuevo_encoded = pd.get_dummies(
    nuevo.astype(str)
)

nuevo_encoded = nuevo_encoded.reindex(
    columns=columnas,
    fill_value=0
)

def interpretar_prediccion(pred):
    valor_original = label_encoder.inverse_transform([pred])[0]

    if valor_original == -1:
        return "Phishing"
    elif valor_original == 0:
        return "Suspicious"
    else:
        return "Legit"

pred_log = log_model.predict(nuevo_encoded)[0]
pred_rf = rf_model.predict(nuevo_encoded)[0]
pred_xgb = xgb_model.predict(nuevo_encoded)[0]

print("\n--- RESULTADOS ---")
print(f"Logistic Regression: {interpretar_prediccion(pred_log)} ({pred_log})")
print(f"Random Forest      : {interpretar_prediccion(pred_rf)} ({pred_rf})")
print(f"XGBoost            : {interpretar_prediccion(pred_xgb)} ({pred_xgb})")

def mostrar_confianza(nombre, modelo):
    proba = modelo.predict_proba(nuevo_encoded)[0]

    print(f"\n{nombre}")

    for clase_codificada, confianza in enumerate(proba):

        valor_original = label_encoder.inverse_transform(
            [clase_codificada]
        )[0]

        if valor_original == -1:
            etiqueta = "Phishing"
        elif valor_original == 0:
            etiqueta = "Suspicious"
        else:
            etiqueta = "Legit"

        print(f"{etiqueta:12}: {confianza:.2%}")

mostrar_confianza(
    "Logistic Regression",
    log_model
)

mostrar_confianza(
    "Random Forest",
    rf_model
)

mostrar_confianza(
    "XGBoost",
    xgb_model
)

Los valores -1 son Legit
Los valores 0 son Suspicious
Los valores 1 son Phishing
SFH [-1, 0, 1]: 1
popUpWindow [-1, 0, 1]: 1
SSLfinal_State [-1, 0, 1]: 1
Request_URL [-1, 0, 1]: 1
URL_of_Anchor [-1, 0, 1]: 1
web_traffic [-1, 0, 1]: 1
URL_Length [-1, 0, 1]: 1
age_of_domain [-1, 1]: 1
having_IP_Address [0, 1]: 1

--- RESULTADOS ---
Logistic Regression: Phishing (0)
Random Forest      : Phishing (0)
XGBoost            : Phishing (0)

Logistic Regression
Phishing    : 99.57%
Suspicious  : 0.43%
Legit       : 0.01%

Random Forest
Phishing    : 99.83%
Suspicious  : 0.00%
Legit       : 0.17%

XGBoost
Phishing    : 99.27%
Suspicious  : 0.17%
Legit       : 0.56%
